# 1. Import Libraries

In [1]:
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

# 2. Load Dataset

In [2]:
df_train = pd.read_excel('../Dataset/training.xlsx')
df_test = pd.read_excel('../Dataset/testing.xlsx')

In [3]:
ANCHOR_YEARS = {1: 2020, 2: 2021, 3: 2022}

In [4]:
time_df = df_train[['Year', 'Month', 'Day', 'Hour']].copy()
time_df['Year'] = time_df['Year'].map(ANCHOR_YEARS)

df_train['Date'] = pd.to_datetime(time_df)
df_train = df_train.sort_values('Date').reset_index(drop=True)

In [5]:
time_df_test = df_test[['Year', 'Month', 'Day', 'Hour']].copy()
time_df_test['Year'] = time_df_test['Year'].map(ANCHOR_YEARS)

df_test['Date'] = pd.to_datetime(time_df_test)
df_test = df_test.sort_values('Date').reset_index(drop=True)

In [6]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Site-1 Temp', 'Site-2 Temp', 'Site-3 Temp', 'Site-4 Temp', 'Site-5 Temp',
    'Site-1 GHI', 'Site-2 GHI', 'Site-3 GHI', 'Site-4 GHI', 'Site-5 GHI'
]
target = 'Load'

# 3. Splitting Data

In [7]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_train['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_train.loc[val_mask, features]
y_val = df_train.loc[val_mask, target]

X_test = df_test[features]
y_test = df_test[target]


print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760
Test (Year 3): 8,760


# 4. Train on year 1, evaluate on year 2

In [8]:
seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,192.429871,37029.253906,141.231628,0.068087,0.775864
1,1,192.429871,37029.253906,141.231628,0.068087,0.775864
2,2,192.429871,37029.253906,141.231628,0.068087,0.775864
3,3,192.429871,37029.253906,141.231628,0.068087,0.775864
4,42,192.429871,37029.253906,141.231628,0.068087,0.775864



=== Stability summary ===


,mean,std,min,max
RMSE,192.429871,0.0,192.429871,192.429871
MSE,37029.253906,0.0,37029.253906,37029.253906
MAE,141.231628,0.0,141.231628,141.231628
MAPE,0.068087,0.0,0.068087,0.068087
R2,0.775864,0.0,0.775864,0.775864


In [12]:
stability_df.to_csv("../report/seed_baseline_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_baseline_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [9]:
print("\n=== RETRAINING ON YEAR 1 + YEAR 2 ===")

train_val_mask = df_train['Year'].isin([2020, 2021])

X_train_full = df_train.loc[
    train_val_mask,
    features
]

y_train_full = df_train.loc[
    train_val_mask,
    target
]


final_model = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_train_full,
    y_train_full
)


=== RETRAINING ON YEAR 1 + YEAR 2 ===


c:\Users\ASUS\anaconda3\envs\pge_energy\lib\site-packages\xgboost\training.py:200: UserWarning: [22:50:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:56: Empty dataset at worker: 0
  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# 5. Final prediction on Test set

In [10]:
X_train_full = pd.concat(
    [X_train, X_val],
    axis=0
)

y_train_full = pd.concat(
    [y_train, y_val],
    axis=0
)

In [11]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train_full,
        y_train_full
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [13]:
pd.DataFrame(y_test_pred).to_csv("../report/baseline_test_prediction.csv", index = False)